# Get on with the Music

A comparison of music listening activity in Springfield and Shelbyville, based on data from an online music streaming service.

## Contents

* Introduction
* Stage 1. Data overview
* Stage 2. Data preprocessing
    * 2.1 Header style
    * 2.2 Missing values
    * 2.3 Duplicates
* Stage 3. Hypothesis testing
* Conclusions

## Introduction

An analyst's job is to extract valuable insights from data so that decisions can be made based on them. That process has several stages, such as a data overview, data preprocessing and hypothesis testing.

Whenever we run a study we need to formulate a hypothesis that we can then test. Sometimes we accept the hypothesis, other times we reject it. To make the right choices, a business must be able to tell whether its assumptions are correct.

In this project we compare the music preferences of the inhabitants of Springfield and Shelbyville. We use data from an online music streaming service to test the hypothesis below and compare the behavior of users in the two cities.

### Objective

Test the hypothesis:

1. User activity differs depending on the day of the week and on the city.

### Stages

User behavior data is stored in the file `music_project_en.csv`. There is no information about the quality of the data, so we need to examine it before testing the hypothesis.

First we assess the quality of the data and check whether its problems are significant. Then, during preprocessing, we deal with the most critical problems.

The project has three stages:

1. Data overview
2. Data preprocessing
3. Hypothesis testing

## Stage 1. Data overview

We open the data and examine it. We only need `pandas` for this project.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('music_project_en.csv')

First 10 rows of the table:

In [3]:
print(df.head(10))

     userID                        Track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

        City        time        Day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

General information about the table:

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0     userID  65079 non-null  str  
 1   Track     63736 non-null  str  
 2   artist    57512 non-null  str  
 3   genre     63881 non-null  str  
 4     City    65079 non-null  str  
 5   time      65079 non-null  str  
 6   Day       65079 non-null  str  
dtypes: str(7)
memory usage: 3.5 MB


### Observations

#### Columns

The table has seven columns, all of them of type `object`. According to the documentation:

* `userID`: user identifier
* `Track`: track title
* `artist`: artist name
* `genre`: music genre
* `City`: the user's city
* `time`: the exact time the track was played
* `Day`: day of the week

#### Header style problems

We can see three style problems in the column names:

1. Some headers are written in uppercase, others in lowercase.
2. Some headers contain spaces (for example `'  userID'` and `'  City  '`).
3. `userID` joins two words without a separator and mixes uppercase and lowercase letters inside the word (camelCase). The convention is snake_case, so it should be `user_id`.

#### Initial assessment

1. **What kind of data do we have, and how do we understand the columns?** All columns are of type `object`. Judging by the documentation and the column names, the data describes tracks played by users from two cities, including the genre and the day of the week, which lets us see the peak days on the platform.
2. **Is the data enough to test the hypothesis?** In principle yes. It only needs to be treated during preprocessing.
3. **Did we notice any problems, such as missing values, duplicates or wrong data types?** Yes. The column names must be standardized and there are missing values in `Track`, `artist` and `genre`. There may also be explicit duplicates and implicit duplicates (the same value written in different ways, for example in `genre`). We use `duplicated()` to count the explicit duplicates and `unique()` to inspect the implicit ones.

## Stage 2. Data preprocessing

The goal here is to prepare the data for the analysis. The first step is to fix the problems in the headers. After that we move on to missing values and duplicates.

### 2.1 Header style

Current column names:

In [5]:
print(df.columns)

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='str')


We change the headers following good style practices:

* All characters in lowercase
* No spaces
* snake_case for names with several words

First, a loop converts every column name to lowercase:

In [6]:
new_columns = []

for old_name in df.columns:  # loop over the headers and convert everything to lowercase
    name_lowered = old_name.lower()
    new_columns.append(name_lowered)

df.columns = new_columns

Then, with the same approach, we remove the spaces at the beginning and at the end of each column name:

In [7]:
new_columns = []

for old_name in df.columns:  # loop over the headers and remove the spaces
    name_no_spaces = old_name.strip()
    new_columns.append(name_no_spaces)

df.columns = new_columns

print(df.columns)

Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')


Finally, `userid` becomes `user_id`, following the snake_case rule:

In [8]:
columns_new_name = {'userid': 'user_id'}

df = df.rename(columns=columns_new_name)

print(df.columns)

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')


Checking the result:

In [9]:
print(df.head(10))

    user_id                        track            artist   genre  \
0  FFB692EC            Kamigata To Boots  The Mass Missile    rock   
1  55204538  Delayed Because of Accident  Andreas Rönnberg    rock   
2    20EC38            Funiculì funiculà       Mario Lanza     pop   
3  A3DD03C9        Dragons in the Sunset        Fire + Ice    folk   
4  E2DC1FAE                  Soul People        Space Echo   dance   
5  842029A1                       Chains          Obladaet  rusrap   
6  4CB90AA5                         True      Roman Messer   dance   
7  F03E1C1F             Feeling This Way   Polina Griffith   dance   
8  8FA1D3BE                     L’estate       Julia Dalia  ruspop   
9  E772D5C0                    Pessimist               NaN   dance   

          city      time        day  
0  Shelbyville  20:28:33  Wednesday  
1  Springfield  14:07:09     Friday  
2  Shelbyville  20:58:07  Wednesday  
3  Shelbyville  08:37:09     Monday  
4  Springfield  08:34:34     Monday  
5

### 2.2 Missing values

First we count the missing values in each column:

In [10]:
print(df.isna().sum())

user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


Not all missing values affect the research. The missing values in `track` and `artist` are not critical, because they can simply be replaced with a default value such as the string `'unknown'`.

Missing values in `genre` could affect the comparison of music preferences in Springfield and Shelbyville. In real life it would be useful to find out why the data is missing and try to correct it, but that is not possible in this project. So we fill these missing values with a default value and then evaluate how much they can affect the analysis.

We replace the missing values in `track`, `artist` and `genre` with `'unknown'`, looping over a list with the names of these columns:

In [11]:
columns_na = ['track', 'artist', 'genre']  # columns where missing values are replaced by 'unknown'

for col in columns_na:
    df[col] = df[col].fillna('unknown')

We count the missing values again to make sure none are left:

In [12]:
print(df.isna().sum())

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


The missing values in `genre` are about 1.8% of the rows (1,198 of 65,079). The hypothesis compares how many tracks are played in each city on each day, not which genres are played, so these rows still count as plays and their effect on the result is negligible.

### 2.3 Duplicates

#### Explicit duplicates

Number of fully duplicated rows:

In [13]:
print(df.duplicated().sum())

3826


We drop all of them:

In [14]:
df = df.drop_duplicates().reset_index(drop=True)

And count again to confirm that none are left:

In [15]:
print(df.duplicated().sum())

0


#### Implicit duplicates in `genre`

Implicit duplicates are values that are written in different ways but mean the same thing, for example a genre name spelled differently. They also affect the results, so we list the unique genre names in alphabetical order and look for them:

In [16]:
print(sorted(df['genre'].unique()))
print()
print(df['genre'].nunique())

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

In the list we find three implicit duplicates of the genre `hiphop`:

* `hip`
* `hop`
* `hip-hop`

To remove them we create the function `replace_wrong_genres()`. It receives the DataFrame, the column name, a list with all the wrong values and the correct value. For each wrong value in the list it applies `replace()` on the column.

In [17]:
wrong_genres = ['hip', 'hop', 'hip-hop']
correct_genre = 'hiphop'


def replace_wrong_genres(df, column, wrong_values, correct_value):
    """Replace every value in `wrong_values` by `correct_value` in `df[column]`."""
    for wrong_value in wrong_values:
        df[column] = df[column].replace(wrong_value, correct_value)
    return df

We call the function to replace `hip`, `hop` and `hip-hop` with `hiphop`. We also unify `türkçe` (Turkish for Turkish) with `türk`, which is another implicit duplicate that we found in the list:

In [18]:
df = replace_wrong_genres(df, 'genre', wrong_genres, correct_genre)
df['genre'] = df['genre'].replace('türkçe', 'türk')

To make sure the duplicated names are gone, we list the unique values of `genre` again:

In [19]:
print(sorted(df['genre'].unique()))
print()
print(df['genre'].nunique())

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

### Observations (preprocessing)

Besides the genres that were pointed out as implicit duplicates (different spellings of `hiphop`), there are other duplicated names, such as `türk` and `türkçe`, which are the same genre, with `türkçe` written in Turkish.

We also found other possible implicit duplicates that raised doubts, such as `trip` and `triphop`, `rock` and `rnr`, and `post`, `postrock` and `posthardcore`. It was not clear whether they are different genres or one genre that includes another, so we left them untouched.

For the two clear cases we used two approaches: `replace()` for `türk` and `türkçe`, and a function for the four spellings of `hiphop`. The `nunique()` method shows the effect: the number of unique genres went down from 269 to 265.

## Stage 3. Hypothesis testing

### Hypothesis: comparing user behavior in the two cities

The hypothesis states that there are differences in how users in Springfield and Shelbyville consume music. To test it, we use the data of three days of the week: Monday, Wednesday and Friday.

* We group the users by city.
* We compare the number of tracks played by each group on Monday, Wednesday and Friday.

Each calculation is done separately, following the split, apply and combine approach: group the data, apply a counting method and get the number of tracks played by each group.

#### Tracks played per city

In [20]:
print(df.groupby('city')['track'].count())

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64


Springfield users played about 24 thousand more tracks than Shelbyville users.

#### Tracks played per day of the week

In [21]:
print(df.groupby('day')['track'].count())

day
Friday       21840
Monday       21354
Wednesday    18059
Name: track, dtype: int64


Friday is the day on which the platform is used the most to listen to music, followed closely by Monday. Wednesday has the lowest number of plays.

#### Tracks played per city and day

Now we need both criteria at the same time. The function `number_tracks()` counts the tracks played on a given day in a given city. It filters the data by day, then filters the result by city, and finally counts the values in the `user_id` column.

In [22]:
def number_tracks(df, week_day, city_name):
    """Number of tracks played on `week_day` in `city_name`."""
    day = df[df['day'] == week_day]  # rows where `day` equals the given weekday
    city = day[day['city'] == city_name]  # rows where `city` equals the given city
    return city['user_id'].count()  # number of values in the user_id column

We call the function for both cities on each of the three days:

In [23]:
for day in ['Monday', 'Wednesday', 'Friday']:
    for city in ['Springfield', 'Shelbyville']:
        print(f'On {day}, {city} had {number_tracks(df, day, city)} tracks played.')

On Monday, Springfield had 15740 tracks played.
On Monday, Shelbyville had 5614 tracks played.
On Wednesday, Springfield had 11056 tracks played.
On Wednesday, Shelbyville had 7003 tracks played.
On Friday, Springfield had 15945 tracks played.
On Friday, Shelbyville had 5895 tracks played.


## Conclusions

The hypothesis is correct: user activity changes during the week and it is clearly different between the two cities.

* Springfield has many more plays than Shelbyville. On some days the difference is above 10 thousand tracks.
* In Springfield the peak is on Friday and the lowest point is on Wednesday.
* In Shelbyville the peak is on Wednesday and the lowest point is on Monday.

So user activity does change depending on the day, but the pattern is not the same in both cities, because the peaks and the lows also depend on the city.

#### Note

In real research projects, statistical hypothesis testing is more precise and quantitative. Conclusions about a whole city cannot always be drawn from the data of a single source.